# Notebook 5: Scikit-Learn Comparison
## Single-Machine Training vs. Distributed PySpark

Train the **same 4 models** on a **smaller sample** using scikit-learn (single machine),  
then compare **accuracy** and **training time** against PySpark MLlib results.

This demonstrates **why Spark scales better** for Big Data workloads.

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from scipy.sparse import hstack

sns.set_theme(style='whitegrid', palette='viridis')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Load Sample Data with Pandas
# ============================================================================
# We use a smaller sample because sklearn runs on a single machine
# This directly demonstrates the scalability limitation

RAW_CSV = os.path.join(PROJECT_ROOT, 'data', 'raw', 'reddit_posts.csv')

print('Loading sample data with pandas...')
print('NOTE: We can only load a fraction because pandas runs on a single machine.')
print('      This is a key limitation compared to PySpark.\n')

start_time = time.time()

# Read only first 200K rows (pandas cannot handle 18GB in memory easily)
SAMPLE_SIZE = 200_000
df = pd.read_csv(RAW_CSV, nrows=SAMPLE_SIZE)

load_time = time.time() - start_time
print(f'Loaded {len(df):,} rows in {load_time:.1f}s')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ============================================================================
# Cell 3: Data Cleaning & Label Creation (same logic as PySpark)
# ============================================================================
# Drop nulls in critical columns
df = df.dropna(subset=['body', 'subreddit', 'normalizedBody'])
df['body'] = df['body'].fillna('')
df['summary'] = df['summary'].fillna('')
df['normalizedBody'] = df['normalizedBody'].fillna('')

# Create virality label (same 80th percentile logic)
subreddit_counts = df['subreddit'].value_counts()
threshold = subreddit_counts.quantile(0.80)
viral_subs = set(subreddit_counts[subreddit_counts >= threshold].index)
df['is_viral'] = df['subreddit'].apply(lambda x: 1 if x in viral_subs else 0)

print(f'Cleaned samples: {len(df):,}')
print(f'Virality threshold: {threshold:.0f} posts')
print(f'Viral subreddits: {len(viral_subs)}')
print(f'\nClass distribution:')
print(df['is_viral'].value_counts())

In [ ]:
# ============================================================================
# Cell 4: Feature Engineering (same features as PySpark pipeline)
# ============================================================================
print('Engineering features...')

# Text features (matching custom transformer)
df['char_count'] = df['body'].str.len()
df['word_count'] = df['body'].str.split().str.len()
df['sentence_count'] = df['body'].str.count(r'[.!?]+')
df['avg_word_length'] = df['char_count'] / df['word_count'].replace(0, 1)
df['question_count'] = df['body'].str.count(r'\?')
df['exclamation_count'] = df['body'].str.count('!')
df['uppercase_ratio'] = df['body'].str.count(r'[A-Z]') / df['char_count'].replace(0, 1)
df['has_url'] = df['body'].str.contains(r'https?://', regex=True).astype(int)
df['paragraph_count'] = df['body'].str.count(r'\n\n') + 1
df['unique_word_ratio'] = df['body'].apply(
    lambda x: len(set(x.lower().split())) / max(len(x.split()), 1)
)

# Summary features
df['summary_length'] = df['summary'].str.len()
df['summary_word_count'] = df['summary'].str.split().str.len()
df['content_density'] = df['summary_length'] / df['char_count'].replace(0, 1)

numeric_features = [
    'char_count', 'word_count', 'sentence_count', 'avg_word_length',
    'unique_word_ratio', 'question_count', 'exclamation_count',
    'uppercase_ratio', 'has_url', 'paragraph_count',
    'summary_length', 'summary_word_count', 'content_density'
]

print(f'Numeric features: {len(numeric_features)}')
df[numeric_features].describe()

In [ ]:
# ============================================================================
# Cell 5: TF-IDF Vectorization
# ============================================================================
print('Building TF-IDF features...')
start_time = time.time()

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    stop_words='english',
    ngram_range=(1, 1)
)

tfidf_matrix = tfidf.fit_transform(df['normalizedBody'])
tfidf_time = time.time() - start_time

print(f'TF-IDF shape: {tfidf_matrix.shape}')
print(f'TF-IDF computation time: {tfidf_time:.1f}s')

# Combine numeric + TF-IDF features
from scipy.sparse import csr_matrix

X_numeric = csr_matrix(df[numeric_features].fillna(0).values)
X = hstack([X_numeric, tfidf_matrix])
y = df['is_viral'].values

print(f'\nFinal feature matrix: {X.shape}')
print(f'Total features: {X.shape[1]} (13 numeric + 5000 TF-IDF)')

In [ ]:
# ============================================================================
# Cell 6: Train-Test Split
# ============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTraining class distribution: {np.bincount(y_train)}')
print(f'Test class distribution:     {np.bincount(y_test)}')

In [ ]:
# ============================================================================
# Cell 7: Train All 4 Models with Scikit-Learn
# ============================================================================
sklearn_models = {
    'Logistic Regression': LogisticRegression(max_iter=100, C=100, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=42)
}

sklearn_results = []

for name, model in sklearn_models.items():
    print(f'\nTraining: {name}')
    print('-' * 40)
    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    metrics = {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_test, y_pred, average='weighted'), 4),
        'F1-Score': round(f1_score(y_test, y_pred, average='weighted'), 4),
        'AUC-ROC': round(roc_auc_score(y_test, y_prob), 4),
        'Training Time (s)': round(train_time, 2)
    }
    sklearn_results.append(metrics)
    
    for k, v in metrics.items():
        print(f'  {k}: {v}')

sklearn_df = pd.DataFrame(sklearn_results)
print('\n' + '=' * 80)
print('SCIKIT-LEARN RESULTS (Single Machine)')
print('=' * 80)
print(sklearn_df.to_string(index=False))

## Spark vs Scikit-Learn Comparison

In [ ]:
# ============================================================================
# Cell 8: Load Spark Results and Compare
# ============================================================================
# Load Spark results from previous notebook
spark_csv = os.path.join(PROJECT_ROOT, 'tableau', 'data', 'spark_model_comparison.csv')
if os.path.exists(spark_csv):
    spark_df = pd.read_csv(spark_csv)
    spark_df['Framework'] = 'PySpark MLlib'
else:
    # Create placeholder if Notebook 4 hasn't been run yet
    spark_df = pd.DataFrame({
        'Model': ['Logistic Regression', 'Random Forest', 'Decision Tree', 'Gradient Boosted Trees'],
        'Accuracy': [0.0, 0.0, 0.0, 0.0],
        'F1-Score': [0.0, 0.0, 0.0, 0.0],
        'Training Time (s)': [0, 0, 0, 0],
        'Framework': ['PySpark MLlib'] * 4
    })
    print('WARNING: Spark results not found. Run Notebook 4 first for full comparison.')

sklearn_df['Framework'] = 'Scikit-Learn'

# Align model names for comparison
sklearn_df['Model'] = sklearn_df['Model'].replace(
    {'Gradient Boosting': 'Gradient Boosted Trees'}
)

# Combine
combined = pd.concat([spark_df, sklearn_df], ignore_index=True)

print('COMBINED COMPARISON')
print('=' * 90)
print(combined[['Framework', 'Model', 'Accuracy', 'F1-Score', 'Training Time (s)']].to_string(index=False))

In [ ]:
# ============================================================================
# Cell 9: Visualization — Spark vs Sklearn
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# F1-Score comparison
sns.barplot(data=combined, x='Model', y='F1-Score', hue='Framework', ax=axes[0], palette=['#2196F3', '#FF5722'])
axes[0].set_title('F1-Score: Spark vs Scikit-Learn', fontsize=13)
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=20)

# Accuracy comparison
sns.barplot(data=combined, x='Model', y='Accuracy', hue='Framework', ax=axes[1], palette=['#2196F3', '#FF5722'])
axes[1].set_title('Accuracy: Spark vs Scikit-Learn', fontsize=13)
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=20)

# Training time comparison
sns.barplot(data=combined, x='Model', y='Training Time (s)', hue='Framework', ax=axes[2], palette=['#2196F3', '#FF5722'])
axes[2].set_title('Training Time: Spark vs Scikit-Learn', fontsize=13)
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'spark_vs_sklearn.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# Cell 10: Analysis — Why Spark Scales Better
# ============================================================================
print('=' * 70)
print('ANALYSIS: Why PySpark Scales Better Than Scikit-Learn')
print('=' * 70)
print('''
1. DATA VOLUME HANDLING:
   - Scikit-learn: Limited to RAM (~200K rows loaded here vs ~3M+ total)
   - PySpark: Processes ALL data across distributed partitions
   - Result: Spark trains on 15-20x more data

2. PARALLEL EXECUTION:
   - Scikit-learn: Single-process (except n_jobs in RF)
   - PySpark: True distributed execution across all cores/executors
   - Result: Near-linear speedup with more resources

3. MEMORY MANAGEMENT:
   - Scikit-learn: Loads entire dataset into RAM (crashes on 18GB)
   - PySpark: Spills to disk, lazy evaluation, partition-level processing
   - Result: Spark handles datasets larger than available RAM

4. FAULT TOLERANCE:
   - Scikit-learn: Job fails → restart from scratch
   - PySpark: RDD lineage allows recovery of lost partitions
   - Result: Spark is production-resilient

5. CONCLUSION:
   Scikit-learn achieves competitive accuracy on small samples, but
   PySpark's distributed architecture is essential for big data (>1GB).
   The quality difference grows as dataset size increases.
''')

# Save combined results for Tableau
combined.to_csv(
    os.path.join(PROJECT_ROOT, 'tableau', 'data', 'spark_vs_sklearn_comparison.csv'),
    index=False
)
print('Results saved to tableau/data/spark_vs_sklearn_comparison.csv')